# Preprocessing smoke test

Проверка всех loader-функций из `src/preprocessing/normalize.py`.

Берём минимальный диапазон и проверяем загрузку в БД

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")
engine = create_engine(os.getenv("DATABASE_URL"))

In [2]:
import subprocess

result = subprocess.run(
    ["psql", os.getenv("DATABASE_URL"), "-f", str(PROJECT_ROOT / "db" / "schema.sql")],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE TABLE
CREATE INDEX
CREATE INDEX
CREATE TABLE

psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:26: NOTICE:  relation "realization_report" already exists, skipping
psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:28: NOTICE:  relation "idx_realization_nm_sale_dt" already exists, skipping
psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:29: NOTICE:  relation "idx_realization_sale_dt" already exists, skipping
psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:39: NOTICE:  relation "warehouses" already exists, skipping
psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:47: NOTICE:  relation "warehouse_remains" already exists, skipping
psql:/Users/macbookair/Documents/wb_logistics_support/db/schema.sql:75: NOTICE:  relation "paid_storage" already exists, skipping
psq

In [ ]:
from src.ingestion.wb_reports import (
    get_warehouses,
    get_warehouse_remains_report,
    get_paid_storage_report,
    get_region_sale,
    get_goods_return,
    get_supplier_sales,
    get_supplier_orders,
    get_report_detail_by_period,
)
from src.preprocessing.normalize import (
    load_warehouses,
    load_warehouse_remains,
    load_paid_storage,
    load_region_sale,
    load_goods_return,
    load_supplier_sales,
    load_supplier_orders,
    load_realization_report,
)

## Smoke test каждого loader'а

In [ ]:
# warehouses (справочник складов WB)
rows = load_warehouses(get_warehouses())
print(f"warehouses: {rows} rows")

In [ ]:
# warehouse_remains (текущий снимок остатков по складам)
rows = load_warehouse_remains(get_warehouse_remains_report(locale="ru", groupByNm=True))
print(f"warehouse_remains: {rows} rows")

In [ ]:
# paid_storage (платное хранение, max 7 дней за запрос)
rows = load_paid_storage(get_paid_storage_report("2026-04-01", "2026-04-07"))
print(f"paid_storage: {rows} rows")

In [ ]:
# region_sale (продажи по регионам)
rows = load_region_sale(get_region_sale("2026-04-01", "2026-04-30"))
print(f"region_sale: {rows} rows")

In [ ]:
# goods_return (возвраты с ПВЗ)
rows = load_goods_return(get_goods_return("2026-04-01", "2026-04-30"))
print(f"goods_return: {rows} rows")

In [ ]:
# supplier_sales (продажи + возвраты, статистика API, max ~6 мес истории)
rows = load_supplier_sales(get_supplier_sales("2026-04-01"))
print(f"supplier_sales: {rows} rows")

In [ ]:
# supplier_orders (заказы, статистика API, max ~6 мес истории)
rows = load_supplier_orders(get_supplier_orders("2026-04-01"))
print(f"supplier_orders: {rows} rows")

In [ ]:
# realization_report (финансовый отчёт, до 3 лет истории- вместо supplier_sales)
rows = load_realization_report(get_report_detail_by_period("2026-04-01", "2026-04-07"))
print(f"realization_report: {rows} rows")

## Итоговая проверка — сколько строк в каждой таблице

In [ ]:
tables = [
    "warehouses", "warehouse_remains", "paid_storage",
    "region_sale", "goods_return",
    "supplier_sales", "supplier_orders", "realization_report",
]
for t in tables:
    n = pd.read_sql(f"SELECT COUNT(*) FROM {t}", engine).iloc[0, 0]
    print(f"{t:25s} {n:>10,} rows")